# Construccion mincer

## Bibliotecas

In [1]:
# Importar bibliotecas
import os
import sys
import warnings

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import pyfixest as pf
import seaborn as sns

warnings.filterwarnings("ignore")

In [2]:
import modulo_500 as empleo
import modulo_300 as educacion

## Ruta de carpetas

In [3]:
# folders
path = "C:/Users/et396/OneDrive/Dropbox/"
base = "Docencia/UNAC/Evaluation/Bases/"
clean = "Docencia/UNAC/Evaluation/S1/Code/"

# Direccion de folder
data_in = path+base
data_out = path+clean

## Modulos

### Modulo Empleo

In [4]:
dataframes_to_concat = [] # diccionario como objeto
# Variables finales 
lista_var =['rcod_hogar','rcod_person','r6','rmujer','redad','rinformal','rDpto','rocu','reduca']

for year in [2021,2022]:
    df = pd.read_stata(data_in + f"enaho01a-{year}-500.dta",
                       convert_categoricals=False)
    
    base = (
        empleo.process500(df)
        .func_hogar()
        .func_person()
        .func_rincome()
        .func_rdpto()
        .func_mujer()
        .func_rage()
        .func_rocu()
        .func_educa()
        .func_informal()
                 )

    data = base[lista_var].copy()
    data['ryear'] = year
    
    dataframes_to_concat.append(data)

empleo_final = pd.concat(dataframes_to_concat, ignore_index=True)    
    
empleo_final.sample(5).T
empleo_final['ryear'].value_counts()

ryear
2022    87661
2021    86806
Name: count, dtype: int64

### Modulo educacion

In [5]:
# Informacion educacion
#  ===============================================
dataframes_to_concat = [] # diccionario como objeto
# Variables finales 
lista_var =['rcod_person','rneduca']

for year in [2021,2022]:
    df = pd.read_stata(data_in + f"enaho01a-{year}-300.dta",
                       convert_categoricals=False)
    
    base = (
        educacion.process300(df)
        .func_hogar()
        .func_person()
        .func_reduca()
                 )

    data = base[lista_var].copy()
    data['ryear'] = year
    
    dataframes_to_concat.append(data)

educacion_final = pd.concat(dataframes_to_concat, ignore_index=True)    
    
educacion_final.sample(5).T
educacion_final['ryear'].value_counts()

ryear
2022    110257
2021    109867
Name: count, dtype: int64

### Union de modulos

In [6]:
# Union de base de datos
# ========================================
base_unida = empleo_final.merge(educacion_final,
                              on=['rcod_person','ryear'],
                              how='left',
                              )
base_unida.sample(6).T

,128911,6621,85861,4331,42565,91522
rcod_hogar,01701402411,00677410611,02028503711,00617110811,01669906811,00692721511
rcod_person,0170140241102,0067741061101,0202850371101,0061711081107,0166990681103,0069272151101
r6,372.0,2748.936686,2299.124837,0.0,385.583333,2193.75
rmujer,1,1,1,1,0,0
redad,55,48,48,14,21,29
rinformal,1,0,0,0,1,0
rDpto,Ica,Ica,Ucayali,Callao,Huanuco,Junin
rocu,1,1,1,3,1,1
reduca,5,3,4,3,3,3
ryear,2022,2021,2021,2021,2021,2022


## Exportar

In [7]:
base_unida.to_csv(data_out +  "BD1.csv", index=False)